# Страница 3 — QLoRA Fine-Tuning: Кавказский говор

Дообучаем **Qwen2.5-7B** (7 миллиардов параметров) на синтетическом датасете кавказского говора.
Используем **QLoRA** = 4-bit квантизация + LoRA адаптер.

| Шаг | Что делаем |
|-----|------------|
| 1 | Загружаем датасет `caucasian_speech.jsonl` |
| 2 | Загружаем Qwen2.5-7B в **4-bit** (QLoRA) |
| 3 | Генерация **до** fine-tuning |
| 4 | Конфигурируем LoRA адаптер |
| 5 | Обучаем (≈20-30 мин на RTX 4060 Ti 16GB) |
| 6 | Генерация **после** + сохранение адаптера |

**VRAM:** ~8-9 GB для модели + ~2 GB обучение = комфортно на 16 GB

In [1]:
import os
# Фикс зависания загрузки HuggingFace на Windows
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_CACHE"] = os.path.abspath("../models/hf_cache")
os.makedirs("../models/hf_cache", exist_ok=True)

from huggingface_hub import whoami
try:
    print("Залогинен как:", whoami()["name"])
except:
    # Залогинься заранее: hf auth login
    print("Не залогинен — запусти в терминале: hf auth login")

print("HF cache:", os.environ["HF_HUB_CACHE"])

In [2]:
# datasets и Arrow импортируем до torch — избегаем конфликта DLL на Windows
from datasets import Dataset as HFDataset
print('datasets ok')

import os, json, time
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
print('transformers + peft ok')

import torch
print(f'torch: {torch.__version__}')

assert torch.cuda.is_available(), 'GPU не найден!'
gpu = torch.cuda.get_device_properties(0)
print(f'GPU: {gpu.name} | VRAM: {gpu.total_memory/1024**3:.1f} GB')

MODELS_DIR   = '../models'
DATA_PATH    = 'caucasian_speech.jsonl'
ADAPTER_PATH = f'{MODELS_DIR}/lora_adapter'
MODEL_NAME   = 'Qwen/Qwen2.5-7B'
device       = torch.device('cuda')
os.makedirs(MODELS_DIR, exist_ok=True)

datasets ok
transformers + peft ok
torch: 2.5.1+cu121
GPU: NVIDIA GeForce RTX 4060 Ti | VRAM: 16.0 GB


## 1. Загрузка датасета

In [3]:
import subprocess, random
if not os.path.exists(DATA_PATH):
    print('Генерируем датасет...')
    subprocess.run(['python', 'generate_dataset.py'], check=True)

texts = [json.loads(l)['text'] for l in open(DATA_PATH, encoding='utf-8')]
print(f'Примеров: {len(texts)}')

random.seed(7)
print('\n--- 3 случайных примера ---')
for t in random.sample(texts, 3):
    print(f'\n• {t}')

Примеров: 1200

--- 3 случайных примера ---

• зачем моросишь, дорогой?

• Вах! дружище Давид! дети растут — радость это, вах! Настоящая радость! жизнью клянусь, к гостю уважение — это самое главное, слышишь? Самое главное! брат позвонил из Тбилиси — соскучился, клянусь, очень соскучился!

• — Ираклий, зачем моросишь, родной?
— Вай вай! в наш дом гость пришёл — всё на стол, всё лучшее, клянусь мамой! вот те крест!
— ты настоящий, это все знают, не газую да?
— Всё, брат. Дружба — это навсегда, слово даю!


## 2. Загрузка Qwen2.5-7B в 4-bit (QLoRA)

In [ ]:
# BitsAndBytesConfig — говорим загрузить модель в 4-bit NormalFloat
# compute_dtype=float16 — вычисления идут в fp16 (быстро на RTX)
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = 'nf4',       # NormalFloat4 — лучше для весов трансформеров
    bnb_4bit_compute_dtype    = torch.float16,
    bnb_4bit_use_double_quant = True,        # двойная квантизация — ещё ~0.4 бит/параметр экономии
)

print(f'Загружаем {MODEL_NAME} в 4-bit...')
print('(первый раз скачает ~15 GB — займёт время)')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token     = tokenizer.eos_token
tokenizer.padding_side  = 'right'   # важно для causal LM

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config = bnb_cfg,
    device_map          = 'auto',   # сам распределит по GPU
    torch_dtype         = torch.float16,
)

n_params = sum(p.numel() for p in base_model.parameters())
print(f'Параметров: {n_params/1e9:.2f}B')
print(f'VRAM занято: {torch.cuda.memory_allocated()/1024**3:.1f} GB')

Загружаем Qwen/Qwen2.5-7B в 4-bit...
(первый раз скачает ~15 GB — займёт время)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

## 3. Генерация ДО fine-tuning

In [ ]:
def generate(model, prompt, max_new_tokens=120, temperature=0.8):
    model.eval()
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens     = max_new_tokens,
            do_sample          = True,
            temperature        = temperature,
            repetition_penalty = 1.2,
            pad_token_id       = tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

TEST_PROMPTS = [
    'Вах, слушай,',
    'Клянусь мамой,',
    'Э, дорогой,',
]

print('=' * 60)
print('  ГЕНЕРАЦИЯ ДО FINE-TUNING')
print('=' * 60)
for p in TEST_PROMPTS:
    print(f'\n[{p}]')
    print(generate(base_model, p))

## 4. LoRA конфигурация

In [ ]:
# prepare_model_for_kbit_training — обязательно для QLoRA:
# включает gradient checkpointing и приводит нормализации к fp32
base_model = prepare_model_for_kbit_training(base_model)

# target_modules для Qwen2.5 — проекции внимания и MLP gate
lora_cfg = LoraConfig(
    task_type      = TaskType.CAUSAL_LM,
    r              = 16,
    lora_alpha     = 32,
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                      'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout   = 0.05,
    bias           = 'none',
)

model = get_peft_model(base_model, lora_cfg)
model.print_trainable_parameters()
# Ожидаем: ~0.5% обучаемых из 7B — примерно 35M параметров

## 5. Токенизация и обучение

In [ ]:
MAX_LEN = 256

def tokenize_fn(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LEN)

dataset  = HFDataset.from_dict({'text': texts})
dataset  = dataset.map(tokenize_fn, batched=True, remove_columns=['text'])
dataset.set_format('torch')

split    = dataset.train_test_split(test_size=0.05, seed=42)
train_ds = split['train']
eval_ds  = split['test']
print(f'Train: {len(train_ds)}  |  Eval: {len(eval_ds)}')

In [ ]:
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

args = TrainingArguments(
    output_dir                  = f'{MODELS_DIR}/lora_ckpt',
    num_train_epochs            = 3,
    per_device_train_batch_size = 4,
    per_device_eval_batch_size  = 8,
    gradient_accumulation_steps = 8,    # эффективный батч = 32
    learning_rate               = 2e-4,
    warmup_ratio                = 0.05,
    weight_decay                = 0.01,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'eval_loss',
    greater_is_better           = False,
    logging_steps               = 20,
    report_to                   = 'none',
    fp16                        = True,
    optim                       = 'paged_adamw_8bit',  # экономит VRAM на оптимизаторе
    dataloader_num_workers      = 0,
    gradient_checkpointing      = True,  # ещё экономия VRAM за счёт скорости
)

trainer = Trainer(
    model         = model,
    args          = args,
    train_dataset = train_ds,
    eval_dataset  = eval_ds,
    data_collator = collator,
)

t0 = time.time()
print('Начинаем QLoRA fine-tuning Qwen2.5-7B...')
trainer.train()
print(f'Готово! Время: {(time.time()-t0)/60:.1f} мин')

## 6. Генерация ПОСЛЕ + сохранение

In [ ]:
print('=' * 60)
print('  СРАВНЕНИЕ: base vs QLoRA')
print('=' * 60)

for p in TEST_PROMPTS:
    print(f'\nПромпт: "{p}"')
    model.disable_adapter_layers()
    print(f'  [BASE] {generate(model, p)}')
    model.enable_adapter_layers()
    print(f'  [LoRA] {generate(model, p)}')

model.enable_adapter_layers()

# Сохраняем только адаптер — несколько сотен МБ вместо 15 GB
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f'\nАдаптер сохранён: {ADAPTER_PATH}')

meta = {
    'base_model'   : MODEL_NAME,
    'adapter_path' : ADAPTER_PATH,
    'lora_r'       : 16,
    'quant'        : '4-bit NF4',
    'train_samples': len(train_ds),
    'test_prompts' : TEST_PROMPTS,
}
with open(f'{MODELS_DIR}/lora_meta.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)
print('Сохранено: lora_meta.json')